<a href="https://www.kaggle.com/code/concacmemay/hybrid-rl-alns-for-vrptw?scriptVersionId=308158408" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# 🚚 Hybrid RL-ALNS for VRPTW — v5
### Application of Reinforcement Learning in Optimizing ALNS for Vehicle Routing with Time Windows

| Algorithm | Description |
|---|---|
| **ALNS** | Baseline — Adaptive Large Neighbourhood Search |
| **OR-Tools** | Strong baseline — Google OR-Tools GLS (60 s, deterministic) |
| **DQN** | Ablation — constructive RL (motivates hybrid design) |
| **RL-ALNS** | 🏆 Proposed — Dueling Double DQN + gating selects operators inside ALNS |
| **RL-ALNS★** | 🏆 Transfer — trained on type-1, zero-shot to type-2 |

**Dataset:** Full Solomon benchmark · 6 families · 56 instances · 100 customers · 3 runs
**New in v5:** RL gating · Reshaped reward (NV-first) · op_counts tracking · Clean session resume

In [ ]:
# ── 1. Install & Imports ──────────────────────────────────────────────────────
!pip install numba safetensors scipy "ortools==9.9.3963" -q

import os, glob, time, random, math, shutil
from collections import deque
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # headless — no display needed on Kaggle
import matplotlib.pyplot as plt
from scipy import stats
from numba import njit

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from safetensors.torch import save_file, load_file

try:
    from ortools.constraint_solver import routing_enums_pb2, pywrapcp
    ORTOOLS_OK = True
except ImportError:
    ORTOOLS_OK = False
    print('⚠ OR-Tools not available')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Device  : {DEVICE}')
print(f'✅ PyTorch : {torch.__version__}')
print(f'✅ OR-Tools: {ORTOOLS_OK}')

In [ ]:
# ── 2. Paths & Config ─────────────────────────────────────────────────────────
IN_KAGGLE  = os.path.exists('/kaggle/working')
DATA_PATH  = ('/kaggle/input/datasets/senju14/vrptw-benchmark-datasets/data/Solomon'
              if IN_KAGGLE else '/content/vrptw-benchmark/data/Solomon')
OUTPUT_DIR = '/kaggle/working' if IN_KAGGLE else '/content'
MODEL_DIR  = os.path.join(OUTPUT_DIR, 'models')
os.makedirs(MODEL_DIR, exist_ok=True)

# ── Session persistence ───────────────────────────────────────────────────────
# After first commit: "+ Add Data" → "Notebook Output Files" → attach this notebook
# Then set PREV_NOTEBOOK_SLUG to the notebook's URL slug (last part of the URL)
PREV_NOTEBOOK_SLUG = ''
PREV_OUTPUT = f'/kaggle/input/{PREV_NOTEBOOK_SLUG}' if PREV_NOTEBOOK_SLUG else ''

PERSISTENT_FILES = [
    'results_main.csv', 'results_transfer.csv',
    'models/transfer_rc1_rc2.safetensors',
    'models/transfer_c1_c2.safetensors',
    'models/transfer_r1_r2.safetensors',
]

def resume_from_previous():
    if not PREV_OUTPUT or not os.path.exists(PREV_OUTPUT):
        print('ℹ Starting fresh — no previous session attached.')
        return
    copied = []
    for fname in PERSISTENT_FILES:
        src = os.path.join(PREV_OUTPUT, fname)
        dst = os.path.join(OUTPUT_DIR, fname)
        if os.path.exists(src) and not os.path.exists(dst):
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            shutil.copy2(src, dst)
            copied.append(fname)
    if copied:
        print(f'✅ Resumed {len(copied)} file(s): {", ".join(copied)}')
    else:
        print('ℹ No new files to resume.')

resume_from_previous()

In [ ]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
@dataclass
class Config:
    # ALNS
    alns_iterations:      int   = 1_000
    rla_iterations:       int   = 1_500
    destroy_ratio_min:    float = 0.10
    destroy_ratio_max:    float = 0.40
    temp_control:         float = 0.05
    temp_decay:           float = 0.99975
    sigma1:               int   = 33
    sigma2:               int   = 9
    sigma3:               int   = 3
    weight_decay:         float = 0.10
    segment_size:         int   = 50
    early_stop_patience:  int   = 350

    # DQN (ablation)
    dqn_state_dim:        int   = 13
    dqn_hidden:           int   = 128
    dqn_lr:               float = 1e-3
    dqn_gamma:            float = 0.95
    dqn_eps_start:        float = 0.80
    dqn_eps_end:          float = 0.05
    dqn_eps_decay:        float = 0.997
    dqn_buffer:           int   = 8_000
    dqn_batch:            int   = 64
    dqn_target_freq:      int   = 20
    dqn_train_freq:       int   = 5
    dqn_vehicle_penalty:  float = 10.0

    # RL-ALNS
    rla_state_dim:        int   = 14
    rla_hidden:           int   = 128
    rla_lr:               float = 1e-3
    rla_gamma:            float = 0.95
    rla_eps_start:        float = 0.40
    rla_eps_end:          float = 0.01
    rla_eps_decay:        float = 0.9997
    rla_buffer:           int   = 8_000
    rla_batch:            int   = 64
    rla_target_freq:      int   = 200
    rla_train_freq:       int   = 10
    rla_prefill:          int   = 200
    # ── v5 reward (NV-first) ──────────────────────────────────────────────────
    rla_reward_vehicle:   float = 15.0   # was 5.0 — NV reduction dominates
    rla_reward_nv_inc:    float = -8.0   # NEW — explicit penalty for NV increase
    rla_reward_cost:      float = 1.0
    rla_reward_bonus:     float = 2.0
    rla_reward_bad:       float = -5.0   # was -3.0
    # ── v5 gating ────────────────────────────────────────────────────────────
    rla_gate_threshold:   float = 0.05   # Q spread below this → defer to ALNS

    # OR-Tools
    ortools_time_s:       int   = 60

    # Experiment
    n_runs:               int   = 3
    seed:                 int   = 42

CFG = Config()

def get_family(name: str) -> str:
    n = name.upper()
    if n.startswith('RC'): return 'RC1' if n[2] == '1' else 'RC2'
    if n.startswith('C'):  return 'C1'  if n[1] == '1' else 'C2'
    if n.startswith('R'):  return 'R1'  if n[1] == '1' else 'R2'
    return 'UNK'

print('✅ Config ready.')

In [ ]:
# ── 4. BKS ────────────────────────────────────────────────────────────────────
BKS = {
    'C101':{'nv':10,'td':828.94},'C102':{'nv':10,'td':828.94},
    'C103':{'nv':10,'td':828.06},'C104':{'nv':10,'td':824.78},
    'C105':{'nv':10,'td':828.94},'C106':{'nv':10,'td':828.94},
    'C107':{'nv':10,'td':828.94},'C108':{'nv':10,'td':828.94},
    'C109':{'nv':10,'td':828.94},
    'C201':{'nv':3,'td':591.56},'C202':{'nv':3,'td':591.56},
    'C203':{'nv':3,'td':591.17},'C204':{'nv':3,'td':590.60},
    'C205':{'nv':3,'td':588.88},'C206':{'nv':3,'td':588.49},
    'C207':{'nv':3,'td':588.29},'C208':{'nv':3,'td':588.32},
    'R101':{'nv':19,'td':1650.80},'R102':{'nv':17,'td':1486.12},
    'R103':{'nv':13,'td':1292.68},'R104':{'nv':9,'td':1007.24},
    'R105':{'nv':14,'td':1377.11},'R106':{'nv':12,'td':1252.03},
    'R107':{'nv':10,'td':1104.66},'R108':{'nv':9,'td':960.88},
    'R109':{'nv':11,'td':1194.73},'R110':{'nv':10,'td':1118.59},
    'R111':{'nv':10,'td':1096.72},'R112':{'nv':9,'td':982.14},
    'R201':{'nv':4,'td':1252.37},'R202':{'nv':3,'td':1191.70},
    'R203':{'nv':3,'td':939.50},'R204':{'nv':2,'td':825.52},
    'R205':{'nv':3,'td':994.42},'R206':{'nv':3,'td':906.14},
    'R207':{'nv':2,'td':890.61},'R208':{'nv':2,'td':726.75},
    'R209':{'nv':3,'td':909.16},'R210':{'nv':3,'td':939.34},
    'R211':{'nv':2,'td':885.71},
    'RC101':{'nv':14,'td':1696.94},'RC102':{'nv':12,'td':1554.75},
    'RC103':{'nv':11,'td':1261.67},'RC104':{'nv':10,'td':1135.48},
    'RC105':{'nv':13,'td':1629.44},'RC106':{'nv':11,'td':1424.73},
    'RC107':{'nv':11,'td':1230.48},'RC108':{'nv':10,'td':1139.82},
    'RC201':{'nv':4,'td':1406.94},'RC202':{'nv':3,'td':1365.64},
    'RC203':{'nv':3,'td':1049.62},'RC204':{'nv':3,'td':798.46},
    'RC205':{'nv':4,'td':1297.65},'RC206':{'nv':3,'td':1146.32},
    'RC207':{'nv':3,'td':1061.14},'RC208':{'nv':3,'td':828.14},
}
print(f'✅ BKS: {len(BKS)} instances across 6 families.')

In [ ]:
# ── 5. Data Loading ───────────────────────────────────────────────────────────
class Inst:
    def __init__(self, raw: Dict):
        self.name          = raw['name']
        data               = raw['data']
        self.capacity      = raw['capacity']
        self.coords        = data[:, 1:3]
        self.demands       = data[:, 3]
        self.ready_times   = data[:, 4]
        self.due_times     = data[:, 5]
        self.service_times = data[:, 6]
        self.horizon       = self.due_times[0]
        self.n             = len(data) - 1
        diff               = self.coords[:, None, :] - self.coords[None, :, :]
        self.dist          = np.sqrt((diff**2).sum(axis=2))
        self.max_dist      = self.dist.max()
        self.tw_width      = self.due_times - self.ready_times
        self.max_tw_width  = self.tw_width[1:].max() + 1e-9
        self.tw_tight_frac = sum(1 for i in range(1, self.n+1)
                                 if self.tw_width[i] < 0.2*self.horizon) / self.n
        d_cust = self.dist[1:, 1:].copy()
        np.fill_diagonal(d_cust, np.inf)
        self.geo_cluster   = float(d_cust.min(axis=1).mean() /
                                   max(self.dist[1:, 1:].mean(), 1e-9))

def _parse(path):
    with open(path) as f: lines = f.readlines()
    return {'name': lines[0].strip(),
            'capacity': float(lines[4].strip().split()[1]),
            'data': np.array([list(map(float, l.split())) for l in lines[9:] if l.strip()])}

def load_solomon(base):
    ds = {}
    for grp, pat in [('c1','c1'),('c2','c2'),('r1','r1'),('r2','r2'),('rc1','rc1'),('rc2','rc2')]:
        files = sorted(glob.glob(os.path.join(base, f'{pat}*.txt')))
        if grp in ('r1','r2'):
            files = [f for f in files if not os.path.basename(f).lower().startswith('rc')]
        ds[grp] = [Inst(_parse(f)) for f in files]
        print(f'  {grp.upper()}: {len(ds[grp])} instances')
    return ds

print('Loading Solomon...')
DS    = load_solomon(DATA_PATH)
C1    = DS['c1'];  C2  = DS['c2']
R1    = DS['r1'];  R2  = DS['r2']
RC1   = DS['rc1']; RC2 = DS['rc2']
ALL   = C1 + C2 + R1 + R2 + RC1 + RC2
print(f'✅ Total: {len(ALL)} instances')

In [ ]:
# ── 6. Solution Model ─────────────────────────────────────────────────────────
@njit(cache=True)
def _rcost(route, dist):
    c = dist[0, route[0]]
    for i in range(len(route)-1): c += dist[route[i], route[i+1]]
    return c + dist[route[-1], 0]

@njit(cache=True)
def _rok(route, demands, cap, ready, due, service, dist):
    load = 0.0
    for n in route: load += demands[n]
    if load > cap: return False
    t, prev = 0.0, 0
    for n in route:
        t += dist[prev, n]
        if t < ready[n]: t = ready[n]
        if t > due[n]:   return False
        t += service[n]; prev = n
    return True

# Pre-compile numba
_d = np.array([1,2], np.int64); _dd = np.zeros((3,3))
_rcost(_d, _dd)
_rok(_d, np.ones(3), 200., np.zeros(3), np.ones(3)*100, np.zeros(3), _dd)
print('✅ Numba compiled.')

class Plan:
    __slots__ = ('routes','inst','_cost','_ok','algo')
    def __init__(self, routes, inst, algo=''):
        self.routes = [r for r in routes if r]
        self.inst = inst; self._cost = None; self._ok = None; self.algo = algo

    @property
    def cost(self):
        if self._cost is None:
            self._cost = sum(_rcost(np.array(r, np.int64), self.inst.dist) for r in self.routes)
        return self._cost

    @property
    def feasible(self):
        if self._ok is None:
            i = self.inst
            self._ok = all(_rok(np.array(r, np.int64), i.demands, i.capacity,
                               i.ready_times, i.due_times, i.service_times, i.dist)
                           for r in self.routes)
        return self._ok

    @property
    def nv(self): return len(self.routes)

    def dominates(self, o):
        return self.nv < o.nv or (self.nv == o.nv and self.cost < o.cost)

    def copy(self): return Plan([r[:] for r in self.routes], self.inst, self.algo)
    def inv(self):  self._cost = self._ok = None

    @property
    def on_time_rate(self):
        i = self.inst; on = total = 0
        for route in self.routes:
            t, prev = 0.0, 0
            for n in route:
                t += i.dist[prev, n]; t = max(t, i.ready_times[n])
                total += 1
                if t <= i.due_times[n]: on += 1
                t += i.service_times[n]; prev = n
        return on / max(total, 1)

    def gap(self):
        bks = BKS.get(self.inst.name, {})
        td = (self.cost - bks['td']) / bks['td'] * 100 if bks else None
        nv = self.nv - bks['nv'] if bks else None
        return td, nv

print('✅ Solution model ready.')

In [ ]:
# ── 7. ALNS Operators ─────────────────────────────────────────────────────────
def _inv(p): p.inv(); return p

def op_random(p, sz):
    nodes = [n for r in p.routes for n in r]
    rem = random.sample(nodes, min(sz, len(nodes))); s = set(rem)
    p.routes = [r for r in [[n for n in r if n not in s] for r in p.routes] if r]
    return _inv(p), rem

def op_worst(p, sz):
    inst = p.inst; gain = []
    for route in p.routes:
        for i, n in enumerate(route):
            prev = route[i-1] if i > 0 else 0; nx = route[i+1] if i < len(route)-1 else 0
            gain.append((inst.dist[prev,n]+inst.dist[n,nx]-inst.dist[prev,nx], n))
    gain.sort(reverse=True); rem = [n for _,n in gain[:sz]]; s = set(rem)
    p.routes = [r for r in [[n for n in r if n not in s] for r in p.routes] if r]
    return _inv(p), rem

def op_shaw(p, sz):
    inst = p.inst; nodes = [n for r in p.routes for n in r]
    if not nodes: return p, []
    seed = random.choice(nodes); rem = [seed]; rs = {seed}
    md = inst.max_dist+1e-9; mt = max(inst.due_times-inst.ready_times)+1e-9
    while len(rem) < sz:
        cands = [(n, 0.5*inst.dist[seed,n]/md
                     + 0.4*abs(inst.ready_times[seed]-inst.ready_times[n])/mt
                     + 0.1*abs(inst.demands[seed]-inst.demands[n])/inst.capacity)
                 for n in nodes if n not in rs]
        if not cands: break
        rem.append(min(cands, key=lambda x: x[1])[0]); rs.add(rem[-1])
    s = set(rem)
    p.routes = [r for r in [[n for n in r if n not in s] for r in p.routes] if r]
    return _inv(p), rem

def op_route(p, sz):
    if len(p.routes) <= 1: return op_random(p, sz)
    rem, to_rm = [], set()
    for idx, route in sorted(enumerate(p.routes), key=lambda x: len(x[1])):
        if len(rem)+len(route) <= sz*1.5: rem.extend(route); to_rm.add(idx)
        if len(rem) >= sz: break
    p.routes = [r for i,r in enumerate(p.routes) if i not in to_rm] or [[]]
    return _inv(p), rem

def op_tw_urgent(p, sz):
    inst = p.inst; nodes = [n for r in p.routes for n in r]
    if not nodes: return p, []
    rem = sorted(nodes, key=lambda n: inst.due_times[n]-inst.ready_times[n])[:sz]
    s = set(rem)
    p.routes = [r for r in [[n for n in r if n not in s] for r in p.routes] if r]
    return _inv(p), rem

def _chk(route, inst):
    return bool(_rok(np.array(route, np.int64), inst.demands, inst.capacity,
                     inst.ready_times, inst.due_times, inst.service_times, inst.dist))

def _bestpos(n, route, inst):
    bc, bp = float('inf'), None
    for p in range(len(route)+1):
        pv = route[p-1] if p > 0 else 0; nx = route[p] if p < len(route) else 0
        c = inst.dist[pv,n]+inst.dist[n,nx]-inst.dist[pv,nx]
        if c < bc and _chk(route[:p]+[n]+route[p:], inst): bc, bp = c, p
    return bc, bp

def _ins(plan, n, inst):
    bc, br, bp = float('inf'), None, None
    for ri, route in enumerate(plan.routes):
        c, p = _bestpos(n, route, inst)
        if p is not None and c < bc: bc, br, bp = c, ri, p
    if br is not None: plan.routes[br].insert(bp, n)
    else:              plan.routes.append([n])
    plan.inv()

def op_greedy(plan, rem):
    inst = plan.inst
    for n in sorted(rem, key=lambda n: inst.due_times[n]): _ins(plan, n, inst)
    return Plan(plan.routes, inst, plan.algo)

def _regret(plan, rem, k):
    inst = plan.inst; rem = rem[:]
    while rem:
        br, chosen, ci = -float('inf'), None, None
        for n in rem:
            opts = sorted((c,ri,p) for ri,r in enumerate(plan.routes)
                          for c,p in [_bestpos(n,r,inst)] if p is not None)
            reg = (sum(opts[i][0]-opts[0][0] for i in range(1,k)) if len(opts)>=k else
                   (opts[1][0]-opts[0][0] if len(opts)>=2 else
                    (float('inf') if len(opts)==1 else -float('inf'))))
            if reg > br and opts: br, chosen, ci = reg, n, opts[0]
        if chosen is not None:
            _, ri, p = ci; plan.routes[ri].insert(p, chosen); plan.inv(); rem.remove(chosen)
        else:
            for n in rem: plan.routes.append([n]); break
    return Plan(plan.routes, inst, plan.algo)

def op_reg2(p, rem): return _regret(p, rem, 2)
def op_reg3(p, rem): return _regret(p, rem, 3)

def op_tw_greedy(plan, rem):
    inst = plan.inst
    for n in sorted(rem, key=lambda n: inst.due_times[n]-inst.ready_times[n]):
        _ins(plan, n, inst)
    return Plan(plan.routes, inst, plan.algo)

DESTROY = [op_random, op_worst, op_shaw, op_route, op_tw_urgent]
REPAIR  = [op_greedy, op_reg2, op_reg3, op_tw_greedy]
N_D, N_R = len(DESTROY), len(REPAIR)
print(f'✅ Operators: {N_D}D × {N_R}R = {N_D*N_R} combos')

In [ ]:
# ── 8. Helpers ────────────────────────────────────────────────────────────────
def build_greedy(inst, algo='') -> Plan:
    customers = sorted(range(1, inst.n+1), key=lambda n: (inst.due_times[n], inst.ready_times[n]))
    unvisited = set(customers); routes = []
    while unvisited:
        route, node, load, t = [], 0, 0.0, 0.0
        while unvisited:
            feasible = [n for n in unvisited
                        if load+inst.demands[n] <= inst.capacity
                        and t+inst.dist[node,n] <= inst.due_times[n]]
            if not feasible: break
            nxt = min(feasible, key=lambda n: inst.dist[node,n])
            route.append(nxt); unvisited.remove(nxt)
            load += inst.demands[nxt]
            t = max(t+inst.dist[node,nxt], inst.ready_times[nxt]) + inst.service_times[nxt]
            node = nxt
        if route: routes.append(route)
    return Plan(routes, inst, algo)

def accept(cur, cand, temp):
    if not cand.feasible: return False
    if cand.nv < cur.nv:  return True
    if cand.nv == cur.nv:
        if cand.cost < cur.cost: return True
        return random.random() < math.exp(-(cand.cost-cur.cost)/max(temp, 1e-6))
    return False

def dsz(it, n_iters, cfg, n):
    ratio = cfg.destroy_ratio_max - (cfg.destroy_ratio_max-cfg.destroy_ratio_min)*(it/n_iters)
    return max(3, int(ratio*n))

print('✅ Helpers ready.')

In [ ]:
# ── 9. Neural Networks ────────────────────────────────────────────────────────
class DQNNet(nn.Module):
    def __init__(self, sd, ad, h):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(sd, h), nn.LayerNorm(h), nn.ReLU(),
            nn.Linear(h, h), nn.ReLU(), nn.Linear(h, ad))
    def forward(self, x): return self.net(x)

class DuelingNet(nn.Module):
    def __init__(self, sd, ad, h):
        super().__init__()
        self.feat = nn.Sequential(nn.Linear(sd, h), nn.LayerNorm(h), nn.ReLU())
        self.val  = nn.Sequential(nn.Linear(h, h//2), nn.ReLU(), nn.Linear(h//2, 1))
        self.adv  = nn.Sequential(nn.Linear(h, h//2), nn.ReLU(), nn.Linear(h//2, ad))
    def forward(self, x):
        f = self.feat(x); v = self.val(f); a = self.adv(f)
        return v + a - a.mean(dim=-1, keepdim=True)

class ReplayBuffer:
    def __init__(self, cap): self.buf = deque(maxlen=cap)
    def push(self, *a): self.buf.append(a)
    def sample(self, k):
        s,a,r,ns,d = zip(*random.sample(self.buf, k))
        return (np.array(s, np.float32), np.array(a, np.int64),
                np.array(r, np.float32), np.array(ns, np.float32), np.array(d, np.float32))
    def __len__(self): return len(self.buf)

print('✅ Networks ready.')

In [ ]:
# ── 10. ALNS Solver ───────────────────────────────────────────────────────────
class ALNSSolver:
    def __init__(self, inst, cfg=CFG):
        self.inst = inst; self.cfg = cfg

    def _roulette(self, w): return int(np.random.choice(len(w), p=w/w.sum()))

    def solve(self, init=None, seed=None):
        if seed is not None: random.seed(seed); np.random.seed(seed)
        cfg = self.cfg
        cur = init.copy() if init else build_greedy(self.inst, 'ALNS'); cur.algo='ALNS'
        best = cur.copy(); temp = cfg.temp_control*cur.cost/math.log(2)
        dw = np.ones(N_D); rw = np.ones(N_R)
        ss = np.zeros((N_D,N_R)); sc = np.zeros((N_D,N_R))
        hist = [best.cost]; no_imp = 0
        for it in range(cfg.alns_iterations):
            di = self._roulette(dw); ri = self._roulette(rw)
            sz = dsz(it, cfg.alns_iterations, cfg, self.inst.n)
            dest, rem = DESTROY[di](cur.copy(), sz); cand = REPAIR[ri](dest, rem)
            score = 0
            if accept(cur, cand, temp):
                if cand.dominates(best):  best=cand.copy(); score=cfg.sigma1; no_imp=0
                elif cand.dominates(cur): score=cfg.sigma2; no_imp=0
                else:                     score=cfg.sigma3; no_imp+=1
                cur = cand
            else: no_imp += 1
            ss[di,ri]+=score; sc[di,ri]+=1
            if (it+1)%cfg.segment_size==0:
                for d in range(N_D):
                    for r in range(N_R):
                        if sc[d,r]>0:
                            avg=ss[d,r]/sc[d,r]
                            dw[d]=dw[d]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                            rw[r]=rw[r]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                ss[:]=0; sc[:]=0; dw=np.maximum(dw,0.1); rw=np.maximum(rw,0.1)
            temp*=cfg.temp_decay; hist.append(best.cost)
            if no_imp>=cfg.early_stop_patience: break
        best.algo='ALNS'; return best, hist

print('✅ ALNS ready.')

In [ ]:
# ── 11. DQN Solver (ablation — shows why pure RL is insufficient) ─────────────
class DQNSolver:
    def __init__(self, inst, cfg=CFG):
        self.inst = inst; self.cfg = cfg
        self.q   = DQNNet(cfg.dqn_state_dim, inst.n+1, cfg.dqn_hidden).to(DEVICE)
        self.q_t = DQNNet(cfg.dqn_state_dim, inst.n+1, cfg.dqn_hidden).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg.dqn_lr)
        self.buf = ReplayBuffer(cfg.dqn_buffer); self.eps = cfg.dqn_eps_start

    def _state(self, node, visited, load, t):
        inst = self.inst; uv = inst.n - len(visited)
        feas = [n for n in range(1, inst.n+1)
                if n not in visited and load+inst.demands[n]<=inst.capacity
                and t+inst.dist[node,n]<=inst.due_times[n]]
        nf = len(feas)
        if feas:
            slacks = [inst.due_times[n]-(t+inst.dist[node,n]) for n in feas]
            ms=min(slacks)/max(inst.horizon,1); av=(sum(slacks)/nf)/max(inst.horizon,1)
            uf=sum(1 for s in slacks if s<0.1*inst.horizon)/max(nf,1)
            aw=(sum(inst.tw_width[n] for n in feas)/nf)/max(inst.max_tw_width,1)
        else: ms=av=uf=aw=0.0
        return np.array([load/inst.capacity, t/max(inst.horizon,1),
                         len(visited)/inst.n, (inst.capacity-load)/inst.capacity,
                         uv/inst.n, nf/max(uv,1),
                         inst.coords[node,0]/100, inst.coords[node,1]/100,
                         inst.demands[node]/inst.capacity,
                         ms, av, uf, aw], dtype=np.float32)

    def _acts(self, node, visited, load, t):
        inst = self.inst; acts = [0]
        for n in range(1, inst.n+1):
            if (n not in visited and load+inst.demands[n]<=inst.capacity
                    and t+inst.dist[node,n]<=inst.due_times[n]): acts.append(n)
        return acts

    def _sel(self, state, feasible):
        if random.random()<self.eps: return random.choice(feasible)
        with torch.no_grad():
            q = self.q(torch.tensor(state).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
        return max(feasible, key=lambda a: q[a])

    def _train(self):
        if len(self.buf)<self.cfg.dqn_batch: return
        s,a,r,ns,d = self.buf.sample(self.cfg.dqn_batch)
        s=torch.tensor(s).to(DEVICE); a=torch.tensor(a,dtype=torch.long).to(DEVICE)
        r=torch.tensor(r).to(DEVICE); ns=torch.tensor(ns).to(DEVICE); d=torch.tensor(d).to(DEVICE)
        qp=self.q(s).gather(1,a.unsqueeze(1)).squeeze(1)
        with torch.no_grad(): tgt=r+self.cfg.dqn_gamma*self.q_t(ns).max(1)[0]*(1-d)
        loss=F.mse_loss(qp,tgt); self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(),1.0); self.opt.step()

    def _episode(self):
        inst=self.inst; visited=set(); routes=[]; trans=[]
        while len(visited)<inst.n:
            route,node,load,t,is_new = [],0,0.0,0.0,True
            while True:
                state=self._state(node,visited,load,t); feas=self._acts(node,visited,load,t)
                if len(feas)==1: break
                action=self._sel(state,feas)
                if action==0: break
                dv=inst.dist[node,action]
                rew=-dv/max(inst.max_dist,1)-(self.cfg.dqn_vehicle_penalty/inst.n if is_new and routes else 0)
                is_new=False; load+=inst.demands[action]
                t=max(t+dv,inst.ready_times[action])+inst.service_times[action]
                visited.add(action); route.append(action)
                ns=self._state(action,visited,load,t); done=(len(visited)==inst.n)
                trans.append((state,action,rew,ns,float(done))); node=action
            if route: routes.append(route)
        return Plan(routes,inst,'DQN'), trans

    def solve(self, seed=None):
        if seed is not None: random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        cfg=self.cfg; best=None; bc=float('inf'); hist=[]; self.eps=cfg.dqn_eps_start
        for ep in range(max(50, cfg.alns_iterations//self.inst.n)):
            plan,trans=self._episode()
            if plan.feasible and trans:
                bonus=max(0,(bc-plan.cost)/bc*10) if bc<float('inf') else 1.0
                s,a,r,ns,d=trans[-1]; trans[-1]=(s,a,r+bonus,ns,d)
                if plan.cost<bc: bc=plan.cost; best=plan.copy()
            for tr in trans: self.buf.push(*tr)
            if ep%cfg.dqn_train_freq==0:
                for _ in range(min(5,len(self.buf)//max(cfg.dqn_batch,1))): self._train()
            if ep%cfg.dqn_target_freq==0: self.q_t.load_state_dict(self.q.state_dict())
            self.eps=max(cfg.dqn_eps_end,self.eps*cfg.dqn_eps_decay)
            hist.append(bc if bc<float('inf') else float('nan'))
        if best is None: best=build_greedy(self.inst,'DQN')
        best.algo='DQN'; return best, hist

print('✅ DQN solver ready.')

In [ ]:
# ── 12. RL-ALNS Solver — Main Contribution ───────────────────────────────────
#
# v5 improvements over v4:
#   1. RL GATING: if Q-value spread < threshold → defer to ALNS adaptive weights
#      This fixes the "blind trust" problem where RL overrides ALNS on hard instances
#   2. NV-FIRST REWARD: rla_reward_vehicle=15 (was 5), explicit penalty for NV increase
#      This fixes RL-ALNS★ using more vehicles than ALNS on RC2
#   3. op_counts: tracks actual operator usage for NEXUS export heatmap

class RLALNSSolver:
    def __init__(self, inst, cfg=CFG, use_dueling=True, feature_mask=None):
        self.inst = inst; self.cfg = cfg
        self.feature_mask = feature_mask
        NetCls = DuelingNet if use_dueling else DQNNet
        self.q   = NetCls(cfg.rla_state_dim, N_D*N_R, cfg.rla_hidden).to(DEVICE)
        self.q_t = NetCls(cfg.rla_state_dim, N_D*N_R, cfg.rla_hidden).to(DEVICE)
        self.q_t.load_state_dict(self.q.state_dict())
        self.opt = optim.Adam(self.q.parameters(), lr=cfg.rla_lr)
        self.buf = ReplayBuffer(cfg.rla_buffer)
        self.eps = cfg.rla_eps_start
        self.op_counts: Dict = {}          # tracks (di,ri) → count for heatmap

    def _state(self, cur, best, it, n_iters, temp, dw, rw, imps):
        inst = self.inst; cfg = self.cfg
        imp_rate  = sum(imps)/len(imps) if imps else 0.0
        cost_gap  = min((cur.cost-best.cost)/max(best.cost,1), 1.0)
        nv_ratio  = cur.nv/max(self._init_nv,1)
        progress  = it/n_iters
        lens  = [len(r) for r in cur.routes] or [0]
        loads = [sum(inst.demands[n] for n in r) for r in cur.routes] or [0]
        rb = float(np.std(lens)/max(np.mean(lens),1)) if len(lens)>1 else 0.0
        lb = float(np.std(loads)/max(inst.capacity,1))
        T0 = cfg.temp_control*best.cost/math.log(2)
        tn = min(temp/max(T0,1e-6),1.0)
        dp = dw/dw.sum(); rp = rw/rw.sum()
        s = np.array([cost_gap, nv_ratio, progress, imp_rate, 1-imp_rate,
                      min(rb,1.0), min(lb,1.0), tn,
                      dp.max(), dp.min(), rp.max(), rp.min(),
                      inst.tw_tight_frac, self._avg_slack(cur)], dtype=np.float32)
        if self.feature_mask is not None:
            s = s * self.feature_mask.astype(np.float32)
        return s

    def _avg_slack(self, plan):
        inst=plan.inst; sl=0.0; n=0
        for route in plan.routes:
            t,prev=0.0,0
            for nd in route:
                t+=inst.dist[prev,nd]; t=max(t,inst.ready_times[nd])
                sl+=inst.due_times[nd]-t; t+=inst.service_times[nd]; prev=nd; n+=1
        return (sl/n)/max(inst.horizon,1) if n else 0.0

    # ── v5 GATING: defer to ALNS when RL is not confident ────────────────────
    def _act(self, state, dw, rw):
        if random.random() < self.eps:
            return random.randrange(N_D*N_R)
        with torch.no_grad():
            q = self.q(torch.tensor(state).unsqueeze(0).to(DEVICE)).cpu().numpy()[0]
        confidence = float(q.max() - q.mean())
        if confidence < self.cfg.rla_gate_threshold:
            # Low confidence → trust ALNS adaptive weights
            di = int(np.random.choice(N_D, p=dw/dw.sum()))
            ri = int(np.random.choice(N_R, p=rw/rw.sum()))
            return di * N_R + ri
        return int(q.argmax())

    # ── v5 REWARD: NV-first ───────────────────────────────────────────────────
    def _reward(self, cur, cand, best):
        cfg = self.cfg
        if not cand.feasible: return cfg.rla_reward_bad
        nv_delta = cur.nv - cand.nv
        r = nv_delta * cfg.rla_reward_vehicle
        if nv_delta < 0:                   # NV increased → extra penalty
            r += cfg.rla_reward_nv_inc
        r += (cur.cost-cand.cost)/max(cur.cost,1)*100*cfg.rla_reward_cost
        if cand.dominates(best): r += cfg.rla_reward_bonus
        return float(r)

    def _train(self):
        if len(self.buf)<self.cfg.rla_batch: return
        s,a,r,ns,d = self.buf.sample(self.cfg.rla_batch)
        s=torch.tensor(s).to(DEVICE); a=torch.tensor(a,dtype=torch.long).to(DEVICE)
        r=torch.tensor(r).to(DEVICE); ns=torch.tensor(ns).to(DEVICE); d=torch.tensor(d).to(DEVICE)
        qp=self.q(s).gather(1,a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            an=self.q(ns).argmax(1)
            qn=self.q_t(ns).gather(1,an.unsqueeze(1)).squeeze(1)
            tgt=r+self.cfg.rla_gamma*qn*(1-d)
        loss=F.mse_loss(qp,tgt); self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(),1.0); self.opt.step()

    def _prefill(self, n_eps):
        cur = build_greedy(self.inst); best = cur.copy()
        self._init_nv = cur.nv
        temp = self.cfg.temp_control*cur.cost/math.log(2)
        dw = np.ones(N_D); rw = np.ones(N_R); imps = deque(maxlen=50)
        for ep in range(n_eps):
            it = ep % max(self.cfg.rla_iterations, 1)
            state = self._state(cur, best, it, self.cfg.rla_iterations, temp, dw, rw, imps)
            action = random.randrange(N_D*N_R); di, ri = action//N_R, action%N_R
            sz = dsz(it, self.cfg.rla_iterations, self.cfg, self.inst.n)
            dest, rem = DESTROY[di](cur.copy(), sz); cand = REPAIR[ri](dest, rem)
            reward = self._reward(cur, cand, best)
            if accept(cur, cand, temp):
                if cand.dominates(best): best=cand.copy()
                cur=cand; imps.append(1)
            else: imps.append(0)
            temp *= self.cfg.temp_decay
            ns_state = self._state(cur, best, it+1, self.cfg.rla_iterations, temp, dw, rw, imps)
            self.buf.push(state, action, reward, ns_state, 0.0)

    def solve(self, seed=None, frozen=False):
        if seed is not None: random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        cfg = self.cfg; n_iters = cfg.rla_iterations
        cur = build_greedy(self.inst, 'RL-ALNS'); best = cur.copy()
        self._init_nv = cur.nv
        temp = cfg.temp_control*cur.cost/math.log(2)
        dw = np.ones(N_D); rw = np.ones(N_R)
        ss = np.zeros((N_D,N_R)); sc = np.zeros((N_D,N_R))
        imps = deque(maxlen=50); hist=[best.cost]; no_imp=0
        if not frozen: self.eps = cfg.rla_eps_start; self.op_counts = {}

        if not frozen and len(self.buf) < cfg.rla_prefill:
            self._prefill(cfg.rla_prefill - len(self.buf))

        for it in range(n_iters):
            state = self._state(cur, best, it, n_iters, temp, dw, rw, imps)
            action = self._act(state, dw, rw)         # ← v5 gating
            di, ri = action//N_R, action%N_R
            # track operator usage for heatmap
            self.op_counts[(di,ri)] = self.op_counts.get((di,ri), 0) + 1

            sz = dsz(it, n_iters, cfg, self.inst.n)
            dest, rem = DESTROY[di](cur.copy(), sz); cand = REPAIR[ri](dest, rem)
            reward = self._reward(cur, cand, best); ascore = 0

            if accept(cur, cand, temp):
                improved = cand.dominates(cur); imps.append(1 if improved else 0)
                if cand.dominates(best):  best=cand.copy(); ascore=cfg.sigma1; no_imp=0
                elif improved:            ascore=cfg.sigma2; no_imp=0
                else:                     ascore=cfg.sigma3; no_imp+=1
                cur=cand
            else: imps.append(0); no_imp+=1

            ss[di,ri]+=ascore; sc[di,ri]+=1
            if (it+1)%cfg.segment_size==0:
                for d in range(N_D):
                    for r in range(N_R):
                        if sc[d,r]>0:
                            avg=ss[d,r]/sc[d,r]
                            dw[d]=dw[d]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                            rw[r]=rw[r]*(1-cfg.weight_decay)+avg*cfg.weight_decay
                ss[:]=0; sc[:]=0; dw=np.maximum(dw,0.1); rw=np.maximum(rw,0.1)

            ns = self._state(cur, best, it+1, n_iters, temp, dw, rw, imps)
            self.buf.push(state, action, reward, ns, float(it==n_iters-1))
            if not frozen:
                if it%cfg.rla_train_freq ==0: self._train()
                if it%cfg.rla_target_freq==0: self.q_t.load_state_dict(self.q.state_dict())
                self.eps = max(cfg.rla_eps_end, self.eps*cfg.rla_eps_decay)
            temp*=cfg.temp_decay; hist.append(best.cost)
            if no_imp>=cfg.early_stop_patience: break

        best.algo = 'RL-ALNS'; return best, hist

    def save(self, path): save_file({k:v.cpu() for k,v in self.q.state_dict().items()}, path)
    def load(self, path):
        self.q.load_state_dict(load_file(path)); self.q_t.load_state_dict(self.q.state_dict())
    def clone_weights(self):
        return {k: v.clone().cpu() for k, v in self.q.state_dict().items()}
    def set_weights(self, w):
        self.q.load_state_dict({k: v.to(DEVICE) for k, v in w.items()})
        self.q_t.load_state_dict(self.q.state_dict())

print('✅ RL-ALNS ready.')

In [ ]:
# ── 13. OR-Tools Solver ───────────────────────────────────────────────────────
def solve_ortools(inst, time_limit_s=60):
    if not ORTOOLS_OK: return build_greedy(inst, 'OR-Tools')
    SCALE = 1000; n_nodes = inst.n+1; n_vehicles = inst.n
    manager = pywrapcp.RoutingIndexManager(n_nodes, n_vehicles, 0)
    routing  = pywrapcp.RoutingModel(manager)
    dist_int = (inst.dist * SCALE).astype(int)
    def _dist(i,j): return int(dist_int[manager.IndexToNode(i), manager.IndexToNode(j)])
    td_cb = routing.RegisterTransitCallback(_dist)
    routing.SetArcCostEvaluatorOfAllVehicles(td_cb)
    def _demand(i): return int(inst.demands[manager.IndexToNode(i)])
    dc = routing.RegisterUnaryTransitCallback(_demand)
    routing.AddDimensionWithVehicleCapacity(dc, 0, [int(inst.capacity)]*n_vehicles, True, 'Cap')
    svc_int = (inst.service_times * SCALE).astype(int)
    def _time(i,j):
        ni = manager.IndexToNode(i)
        return int(dist_int[ni, manager.IndexToNode(j)]) + int(svc_int[ni])
    tc = routing.RegisterTransitCallback(_time)
    horizon = int(inst.horizon * SCALE)
    routing.AddDimension(tc, horizon, horizon, False, 'Time')
    td = routing.GetDimensionOrDie('Time')
    for node in range(1, n_nodes):
        idx = manager.NodeToIndex(node)
        td.CumulVar(idx).SetRange(int(inst.ready_times[node]*SCALE), int(inst.due_times[node]*SCALE))
    sp = pywrapcp.DefaultRoutingSearchParameters()
    sp.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PARALLEL_CHEAPEST_INSERTION
    sp.local_search_metaheuristic = routing_enums_pb2.LocalSearchMetaheuristic.GUIDED_LOCAL_SEARCH
    sp.time_limit.seconds = time_limit_s; sp.log_search = False
    sol = routing.SolveWithParameters(sp)
    if sol is None: return build_greedy(inst, 'OR-Tools')
    routes = []
    for v in range(n_vehicles):
        route=[]; idx=routing.Start(v)
        while not routing.IsEnd(idx):
            node=manager.IndexToNode(idx)
            if node != 0: route.append(node)
            idx=sol.Value(routing.NextVar(idx))
        if route: routes.append(route)
    plan = Plan(routes, inst, 'OR-Tools')
    return plan if plan.feasible else build_greedy(inst, 'OR-Tools')

print('✅ OR-Tools solver ready.')

In [ ]:
# ── 14. Benchmark Runner ──────────────────────────────────────────────────────
def run_instance(inst, algo, cfg, seed, transfer_weights=None):
    t0 = time.time()
    if algo == 'ALNS':
        plan, hist = ALNSSolver(inst, cfg).solve(seed=seed)
    elif algo == 'DQN':
        plan, hist = DQNSolver(inst, cfg).solve(seed=seed)
    elif algo == 'RL-ALNS':
        solver = RLALNSSolver(inst, cfg)
        plan, hist = solver.solve(seed=seed)
    elif algo == 'RL-ALNS★':
        solver = RLALNSSolver(inst, cfg)
        if transfer_weights is not None:
            solver.set_weights(transfer_weights)
            solver.eps = cfg.rla_eps_end
        plan, hist = solver.solve(seed=seed, frozen=True)
        plan.algo = 'RL-ALNS★'
    elif algo == 'OR-Tools':
        plan = solve_ortools(inst, cfg.ortools_time_s)
        hist = [plan.cost]
    else:
        raise ValueError(algo)
    elapsed = time.time() - t0
    bks = BKS.get(inst.name, {})
    return {
        'nv': plan.nv, 'cost': plan.cost, 'time': elapsed,
        'td_gap': (plan.cost-bks['td'])/bks['td']*100 if bks.get('td') else None,
        'nv_diff': plan.nv - bks['nv'] if bks.get('nv') else None,
        'on_time': plan.on_time_rate, 'hist': hist,
    }

def run_benchmark(instances, algorithms, cfg=CFG, result_path=None, transfer_weights=None):
    if result_path is None:
        result_path = os.path.join(OUTPUT_DIR, 'benchmark.csv')
    done = set()
    if os.path.exists(result_path):
        ex = pd.read_csv(result_path)
        done = set(zip(ex['Instance'], ex['Algorithm']))
        print(f'  Resuming — {len(done)} combos already done')
    total = len(instances)*len(algorithms)
    print(f'  Total: {total} combos × {cfg.n_runs} runs\n' + '='*60)
    for inst in instances:
        ds = get_family(inst.name)
        for algo in algorithms:
            if (inst.name, algo) in done:
                print(f'  ⏭  {inst.name} {algo}'); continue
            print(f'\n  [{inst.name}] {algo}')
            nvs,costs,times,gaps,nvds,ots = [],[],[],[],[],[]
            n_reps = 1 if algo == 'OR-Tools' else cfg.n_runs
            for run in range(n_reps):
                res = run_instance(inst, algo, cfg, cfg.seed+run, transfer_weights)
                nvs.append(res['nv']); costs.append(res['cost'])
                times.append(res['time']); gaps.append(res['td_gap'])
                nvds.append(res['nv_diff']); ots.append(res['on_time'])
                print(f'    run {run+1}/{n_reps}: nv={res["nv"]} '
                      f'cost={res["cost"]:.1f} ({res["time"]:.1f}s)')
            row = pd.DataFrame([{
                'Dataset': ds, 'Instance': inst.name, 'Algorithm': algo,
                'NV_mean':  round(np.mean(nvs),2),
                'NV_std':   round(np.std(nvs), 2),
                'NV_diff':  round(np.mean(nvds),2) if nvds[0] is not None else None,
                'TD_mean':  round(np.mean(costs),2),
                'TD_std':   round(np.std(costs),2),
                'Gap%':     round(np.mean(gaps),2) if gaps[0] is not None else None,
                'OnTime':   round(np.mean(ots)*100,1),
                'Time_s':   round(np.mean(times),1),
                'NV_cv':    round(np.std(nvs)/max(np.mean(nvs),1)*100,2),
                'TD_cv':    round(np.std(costs)/max(np.mean(costs),1)*100,2),
            }])
            row.to_csv(result_path, mode='a',
                       header=not os.path.exists(result_path), index=False)
            gv = f'{np.mean(gaps):+.1f}%' if gaps[0] is not None else '—'
            print(f'  → nv={np.mean(nvs):.1f}±{np.std(nvs):.1f}  '
                  f'td={np.mean(costs):.1f}±{np.std(costs):.1f}  gap={gv}')
    return pd.read_csv(result_path)

print('✅ Benchmark runner ready.')

In [ ]:
# ── 15. Transfer Learning ─────────────────────────────────────────────────────
def train_transfer(instances, cfg=CFG, seed=42, tag='transfer'):
    print(f'📚 Training transfer model [{tag}]...')
    all_weights = []
    for inst in instances:
        print(f'  {inst.name}...', end=' ')
        solver = RLALNSSolver(inst, cfg)
        plan, _ = solver.solve(seed=seed)
        all_weights.append(solver.clone_weights())
        td, nv = plan.gap()
        print(f'nv={plan.nv}, gap={td:+.1f}%' if td else f'nv={plan.nv}')
    keys = all_weights[0].keys()
    averaged = {k: torch.stack([w[k].float() for w in all_weights]).mean(0) for k in keys}
    path = os.path.join(MODEL_DIR, f'transfer_{tag}.safetensors')
    save_file(averaged, path)
    print(f'✅ Saved → {path}')
    return averaged

def load_transfer(tag='transfer'):
    path = os.path.join(MODEL_DIR, f'transfer_{tag}.safetensors')
    if os.path.exists(path):
        print(f'✅ Loaded transfer model: {tag}')
        return load_file(path)
    return None

print('✅ Transfer pipeline ready.')

In [ ]:
# ── 16. Statistics ────────────────────────────────────────────────────────────
def wilcoxon_test(df, algo_a, algo_b, metric='Gap%', dataset=None):
    sub = df if dataset is None else df[df['Dataset']==dataset]
    a = sub[sub['Algorithm']==algo_a][metric].dropna().values
    b = sub[sub['Algorithm']==algo_b][metric].dropna().values
    n = min(len(a), len(b)); a, b = a[:n], b[:n]
    if n < 3: return {'stat': None, 'p': None, 'sig': False, 'n': n}
    stat, p = stats.wilcoxon(a, b, alternative='two-sided')
    # Cohen's d (effect size)
    diff = a - b; d = diff.mean() / max(diff.std(), 1e-9)
    return {'stat': round(stat,3), 'p': round(p,4), 'sig': p<0.05, 'n': n,
            'better': algo_a if a.mean()<b.mean() else algo_b,
            'effect_d': round(d, 3)}

def print_results_table(df):
    summary = (
        df.groupby(['Dataset','Algorithm'])
          .agg(NV=('NV_mean','mean'), NV_std=('NV_std','mean'), NV_d=('NV_diff','mean'),
               TD=('TD_mean','mean'), TD_std=('TD_std','mean'), Gap=('Gap%','mean'),
               CV_nv=('NV_cv','mean'), CV_td=('TD_cv','mean'),
               OT=('OnTime','mean'), Time=('Time_s','mean'))
          .round(2).reset_index()
    )
    hdr = f'{"DS":<5}{"Algorithm":<12}{"NV":>6}{"±":>4}{"vBKS":>6}{"TD":>9}{"±":>6}{"Gap%":>7}{"CV_NV":>6}{"CV_TD":>6}{"OT%":>6}{"Time":>7}'
    print('\n' + '─'*len(hdr)); print(hdr); print('─'*len(hdr))
    prev = ''
    for _, r in summary.iterrows():
        if r['Dataset'] != prev and prev: print('─'*len(hdr))
        prev = r['Dataset']
        nv_d = f"{r['NV_d']:+.1f}" if pd.notna(r['NV_d']) else '—'
        gap  = f"{r['Gap']:+.1f}%" if pd.notna(r['Gap']) else '—'
        print(f"{r['Dataset']:<5}{r['Algorithm']:<12}"
              f"{r['NV']:>6.1f}{r['NV_std']:>4.1f}{nv_d:>6}"
              f"{r['TD']:>9.1f}{r['TD_std']:>6.1f}{gap:>7}"
              f"{r['CV_nv']:>6.1f}{r['CV_td']:>6.1f}"
              f"{r['OT']:>6.1f}{r['Time']:>6.1f}s")
    print('─'*len(hdr))

def print_stats_table(df):
    print('\n── Statistical Significance (Wilcoxon signed-rank + Cohen\'s d) ──')
    hdr = f'{"Comparison":<28}{"DS":<5}{"Metric":<8}{"W":>7}{"p":>8}{"Sig":>5}{"d":>7}{"Better":>12}'
    print(hdr); print('─'*len(hdr))
    algos = df['Algorithm'].unique()
    pairs = [('RL-ALNS','ALNS'),('OR-Tools','ALNS'),('RL-ALNS★','ALNS')]
    pairs = [(a,b) for a,b in pairs if a in algos and b in algos]
    for a,b in pairs:
        for ds in df['Dataset'].unique():
            for metric in ['Gap%','NV_mean']:
                r = wilcoxon_test(df, a, b, metric, ds)
                if r['stat'] is None: continue
                sig = '✅' if r['sig'] else '—'
                print(f'  {a} vs {b:<6}  {ds:<5}{metric:<8}'
                      f'{r["stat"]:>7.1f}{r["p"]:>8.4f}{sig:>5}'
                      f'{r["effect_d"]:>7.3f}{r["better"]:>12}')
    print('─'*len(hdr))
    print('✅ = p<0.05  |  d = Cohen\'s d effect size')

print('✅ Statistics ready.')

In [ ]:
# ── 17. Visualisation ─────────────────────────────────────────────────────────
COLORS = {'ALNS':'#5f5fae','DQN':'#e8593c','RL-ALNS':'#1d9e75',
          'RL-ALNS★':'#f2a623','OR-Tools':'#9b59b6'}
HATCH  = {'ALNS':'','DQN':'//','RL-ALNS':'','RL-ALNS★':'xx','OR-Tools':'..'}
FAMILIES = ['C1','C2','R1','R2','RC1','RC2']

def _savefig(name):
    plt.savefig(os.path.join(OUTPUT_DIR, name), dpi=150, bbox_inches='tight')
    plt.close('all')

def plot_family_summary(df):
    fams  = [f for f in FAMILIES if f in df['Dataset'].values]
    algos = [a for a in COLORS if a in df['Algorithm'].values]
    x = np.arange(len(fams)); w = 0.75/max(len(algos),1)
    fig, ax = plt.subplots(figsize=(12,5))
    for ji, algo in enumerate(algos):
        vals = [df[(df['Dataset']==f)&(df['Algorithm']==algo)]['Gap%'].mean()
                for f in fams]
        ax.bar(x+ji*w, vals, w, label=algo, color=COLORS[algo],
               hatch=HATCH[algo], alpha=0.85, edgecolor='white')
    ax.set_xticks(x+w*(len(algos)-1)/2); ax.set_xticklabels(fams)
    ax.set_ylabel('Mean Gap% vs BKS (↓)'); ax.axhline(0, color='gray', lw=0.5, ls='--')
    ax.set_title('Gap% by Solomon Family — v5', fontweight='bold')
    ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
    _savefig('family_summary.png'); print('✅ Saved family_summary.png')

def plot_dashboard(df):
    fams  = [f for f in FAMILIES if f in df['Dataset'].values]
    algos = [a for a in COLORS if a in df['Algorithm'].values]
    metrics = [('Gap%','Gap% vs BKS','↓'),('NV_mean','Vehicles used','↓'),('TD_cv','TD CV%','↓ stable')]
    fig, axes = plt.subplots(len(fams), 3, figsize=(18, 3.5*len(fams)))
    if len(fams)==1: axes=[axes]
    for ri, ds in enumerate(fams):
        for ci,(met,lbl,note) in enumerate(metrics):
            ax = axes[ri][ci]; sub = df[df['Dataset']==ds]
            if sub.empty: continue
            insts = sub['Instance'].unique(); x = np.arange(len(insts)); w = 0.18
            for ji, algo in enumerate(algos):
                vals = [sub[sub['Instance']==i][met].mean() for i in insts]
                ax.bar(x+ji*w, vals, w, label=algo, color=COLORS[algo],
                       hatch=HATCH[algo], alpha=0.85, edgecolor='white')
            ax.set_xticks(x+w*(len(algos)-1)/2)
            ax.set_xticklabels([i.upper()[-3:] for i in insts], fontsize=7)
            ax.set_title(f'{ds} — {lbl} ({note})', fontsize=8, fontweight='bold')
            ax.set_ylabel(met, fontsize=8); ax.grid(axis='y', alpha=0.3)
            if ri==0 and ci==0: ax.legend(fontsize=7)
    plt.suptitle('Algorithm Comparison — VRPTW Solomon v5', fontsize=13, fontweight='bold')
    plt.tight_layout(); _savefig('dashboard.png'); print('✅ Saved dashboard.png')

def plot_transfer(df, src_fam, tgt_fam, algo='RL-ALNS★'):
    if algo not in df['Algorithm'].values: return
    sub = df[df['Dataset']==tgt_fam]
    alns = sub[sub['Algorithm']=='ALNS'].set_index('Instance')['Gap%']
    rla  = sub[sub['Algorithm']==algo].set_index('Instance')['Gap%']
    common = alns.index.intersection(rla.index)
    if common.empty: return
    fig, ax = plt.subplots(figsize=(7,5))
    for nm in common:
        a, r = alns[nm], rla[nm]
        ax.scatter(a, r, color='#1d9e75' if r<a else '#e8593c', s=80, zorder=3)
        ax.annotate(nm[-3:].upper(), (a,r), textcoords='offset points', xytext=(5,3), fontsize=8)
    lo = min(alns[common].min(), rla[common].min())-1
    hi = max(alns[common].max(), rla[common].max())+1
    ax.plot([lo,hi],[lo,hi],'k--',lw=1,label='equal')
    ax.set_xlabel('ALNS Gap%'); ax.set_ylabel(f'{algo} Gap%')
    ax.set_title(f'Transfer {src_fam}→{tgt_fam}: below diagonal = transfer wins', fontweight='bold')
    ax.legend(); ax.grid(alpha=0.2); plt.tight_layout()
    _savefig(f'transfer_{src_fam}_{tgt_fam}.png')
    print(f'✅ Saved transfer_{src_fam}_{tgt_fam}.png')

def plot_routes(plan, save_name=None):
    RCOLS = ['#E63946','#2A9D8F','#E9C46A','#264653','#F4A261',
             '#A8DADC','#457B9D','#6A4C93','#F72585','#4CC9F0',
             '#80B918','#FF9F1C','#8338EC','#3A86FF','#CBFF8C']
    inst=plan.inst; fig,ax=plt.subplots(figsize=(9,7))
    ax.scatter(*inst.coords[0], s=220, c='black', marker='s', zorder=5)
    for i, route in enumerate(plan.routes):
        col=RCOLS[i%len(RCOLS)]; stops=[0]+route+[0]
        xs=[inst.coords[n,0] for n in stops]; ys=[inst.coords[n,1] for n in stops]
        ax.plot(xs,ys,'-o',color=col,lw=1.5,ms=4,alpha=0.8,label=f'V{i+1}')
    td,nv=plan.gap(); g=f' | BKS: TD {td:+.1f}% NV {nv:+d}' if td else ''
    ax.set_title(f'{plan.algo} — {inst.name}  nv={plan.nv}  cost={plan.cost:.1f}{g}',
                 fontweight='bold')
    ax.legend(fontsize=6,ncol=3); ax.grid(alpha=0.2); plt.tight_layout()
    fname = save_name or f'routes_{inst.name}_{plan.algo}.png'
    _savefig(fname); print(f'✅ Saved {fname}')

print('✅ Visualisation ready.')

In [ ]:
# ── 18. Smoke Test ────────────────────────────────────────────────────────────
smoke = Config(alns_iterations=150, rla_iterations=250,
               early_stop_patience=80, n_runs=1, rla_prefill=50, ortools_time_s=5)
inst = RC1[0]
print(f'Smoke test — {inst.name}\n')
for algo, Cls in [('ALNS', ALNSSolver), ('RL-ALNS', RLALNSSolver)]:
    t0=time.time(); s=Cls(inst, smoke); plan,_=s.solve(seed=42); el=time.time()-t0
    td,nv=plan.gap()
    print(f'  {algo:<12} nv={plan.nv:>3}  cost={plan.cost:>8.1f}  '
          f'BKS TD {td:+.1f}% NV {nv:+d}  ({el:.1f}s)')
if ORTOOLS_OK:
    t0=time.time(); p=solve_ortools(inst,5); el=time.time()-t0
    td,nv=p.gap()
    print(f'  {"OR-Tools":<12} nv={p.nv:>3}  cost={p.cost:>8.1f}  '
          f'BKS TD {td:+.1f}% NV {nv:+d}  ({el:.1f}s)')
print('\n✓ Smoke test passed')

In [ ]:
# ── 19. Phase 1 — All 56 Solomon instances: ALNS + DQN + RL-ALNS ─────────────
#
# Runtime estimate (T4 x2, n_runs=3):
#   56 inst × ALNS (~45s)   = ~42 min
#   56 inst × DQN  (~5s)    = ~5 min
#   56 inst × RL-ALNS (~120s) = ~112 min
#   Total phase 1 ≈ ~2.6h
#
resume_from_previous()
RESULT_MAIN = os.path.join(OUTPUT_DIR, 'results_main.csv')

df_main = run_benchmark(
    instances   = ALL,
    algorithms  = ['ALNS', 'DQN', 'RL-ALNS'],
    cfg         = CFG,
    result_path = RESULT_MAIN,
)
print_results_table(df_main)

---
> 💾 **Checkpoint 1** — commit now, then attach output and set PREV_NOTEBOOK_SLUG.

In [ ]:
# ── 20. Phase 2 — OR-Tools baseline (all 56 instances, 1 run each) ────────────
#
# Runtime: 56 inst × 60s = ~56 min
#
RESULT_ORT = os.path.join(OUTPUT_DIR, 'results_ortools.csv')

df_ort = run_benchmark(
    instances   = ALL,
    algorithms  = ['OR-Tools'],
    cfg         = CFG,
    result_path = RESULT_ORT,
)

# Merge with main for combined table
df_p2 = pd.concat([df_main, df_ort], ignore_index=True)
print_results_table(df_p2)
print_stats_table(df_p2)

In [ ]:
# ── 21. Phase 3 — Transfer Learning ──────────────────────────────────────────
#
# 3 transfer scenarios:
#   RC1 → RC2 (same geo, TW widens)
#   C1  → C2  (clustered, TW widens)
#   R1  → R2  (random, TW widens)
#
# Runtime: 3 × (train ~1h + eval ~20min) ≈ ~3.7h total
#
resume_from_previous()
RESULT_TR = os.path.join(OUTPUT_DIR, 'results_transfer.csv')

TRANSFER_PAIRS = [
    (RC1, RC2, 'rc1_rc2', 'RC2'),
    (C1,  C2,  'c1_c2',   'C2'),
    (R1,  R2,  'r1_r2',   'R2'),
]

all_transfer_dfs = []
for src_insts, tgt_insts, tag, tgt_fam in TRANSFER_PAIRS:
    print(f'\n{"="*60}\nTransfer: {tag}')
    tw = load_transfer(tag)
    if tw is None:
        tw = train_transfer(src_insts, CFG, seed=CFG.seed, tag=tag)

    result_path_tr = os.path.join(OUTPUT_DIR, f'results_transfer_{tag}.csv')
    df_tr = run_benchmark(
        instances        = tgt_insts,
        algorithms       = ['RL-ALNS★'],
        cfg              = CFG,
        result_path      = result_path_tr,
        transfer_weights = tw,
    )
    all_transfer_dfs.append(df_tr)

    # Comparison plot
    df_src_base = df_main[df_main['Dataset'].isin([tgt_fam])]
    df_compare  = pd.concat([df_src_base, df_tr], ignore_index=True)
    src_fam = tag.split('_')[0].upper()
    plot_transfer(df_compare, src_fam, tgt_fam)

# Consolidate transfer results
df_transfer = pd.concat(all_transfer_dfs, ignore_index=True)
df_transfer.to_csv(RESULT_TR, index=False)

# Full combined table
df_full = pd.concat([df_main, df_ort, df_transfer], ignore_index=True)
print('\n' + '='*60 + '\nFULL RESULTS TABLE')
print_results_table(df_full)
print_stats_table(df_full)

---
> 💾 **Checkpoint 2** — commit, then resume for Phase 4.

In [ ]:
# ── 22. Phase 4 — Ablation Study (RC1 only, 2 seeds) ─────────────────────────
#
# 4a Architecture: DuelingNet vs plain DQNNet
# 4b Features:     Full / No TW / No weights / No balance
# 4c Prefill:      0 / 100 / 200(default) / 500 / 1000
#
# Runtime: ~1.2h on RC1 × 2 seeds
#
ABL_CFG = Config(alns_iterations=600, rla_iterations=900,
                 n_runs=2, rla_prefill=200, early_stop_patience=250)
ABL_SEEDS = [CFG.seed, CFG.seed+1]
ABL_INSTS = RC1   # 8 instances

# 4a — Architecture
abl_arch = []
for inst in ABL_INSTS:
    for label, dueling in [('Dueling (proposed)', True), ('Plain DQN', False)]:
        gaps = []
        for seed in ABL_SEEDS:
            s = RLALNSSolver(inst, ABL_CFG, use_dueling=dueling)
            plan, _ = s.solve(seed=seed)
            td, _ = plan.gap()
            if td is not None: gaps.append(td)
        abl_arch.append({'Instance': inst.name, 'Variant': label,
                         'Gap%_mean': round(np.mean(gaps),2),
                         'Gap%_std':  round(np.std(gaps),2)})

df_abl_arch = pd.DataFrame(abl_arch)
print('\n── Ablation 4a: Architecture ──')
print(df_abl_arch.pivot_table(index='Instance', columns='Variant',
                               values='Gap%_mean').round(2).to_string())

fig, ax = plt.subplots(figsize=(10,4))
variants = df_abl_arch['Variant'].unique(); x = np.arange(len(ABL_INSTS)); w = 0.35
for ji, var in enumerate(variants):
    sub = df_abl_arch[df_abl_arch['Variant']==var]
    ax.bar(x+ji*w, sub['Gap%_mean'], w, yerr=sub['Gap%_std'],
           label=var, alpha=0.85, capsize=3,
           color=['#1d9e75','#e8593c'][ji])
ax.set_xticks(x+w/2); ax.set_xticklabels([i.name[-3:] for i in ABL_INSTS])
ax.set_ylabel('Gap% vs BKS (↓)'); ax.set_title('Ablation 4a: Dueling vs Plain DQN', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3); plt.tight_layout()
_savefig('ablation_arch.png'); print('✅ Saved ablation_arch.png')

# 4b — State features
FEAT_VARIANTS = {
    'Full (baseline)':  np.ones(14, dtype=bool),
    'No TW features':   np.array([1]*12+[0,0], dtype=bool),
    'No weight feats':  np.array([1]*8+[0,0,0,0,1,1], dtype=bool),
    'No balance feats': np.array([1]*5+[0,0]+[1]*7, dtype=bool),
}
abl_feat = []
for inst in ABL_INSTS[:4]:  # 4 instances for speed
    for label, mask in FEAT_VARIANTS.items():
        gaps = []
        for seed in ABL_SEEDS:
            s = RLALNSSolver(inst, ABL_CFG, feature_mask=mask)
            plan, _ = s.solve(seed=seed)
            td, _ = plan.gap()
            if td is not None: gaps.append(td)
        abl_feat.append({'Instance': inst.name, 'Variant': label,
                         'Gap%_mean': round(np.mean(gaps),2),
                         'Gap%_std':  round(np.std(gaps),2)})

df_abl_feat = pd.DataFrame(abl_feat)
feat_summary = df_abl_feat.groupby('Variant')['Gap%_mean'].agg(['mean','std']).round(2)
print('\n── Ablation 4b: Feature importance ──')
print(feat_summary)

fig, ax = plt.subplots(figsize=(8,4))
labels = list(feat_summary.index); means = feat_summary['mean'].values; stds = feat_summary['std'].values
colors_f = ['#1d9e75','#f2a623','#5f5fae','#e8593c']
ax.barh(labels, means, xerr=stds, color=colors_f, alpha=0.85, capsize=4)
ax.axvline(means[0], color='gray', ls='--', lw=1, label='Full baseline')
ax.set_xlabel('Mean Gap% vs BKS (↓)'); ax.set_title('Ablation 4b: Feature Importance', fontweight='bold')
ax.legend(); ax.grid(axis='x', alpha=0.3); plt.tight_layout()
_savefig('ablation_features.png'); print('✅ Saved ablation_features.png')

# 4c — Prefill size
PREFILL_SIZES = [0, 100, 200, 500, 1000]
abl_pf = []
for inst in ABL_INSTS[:3]:  # 3 instances
    for pf in PREFILL_SIZES:
        cfg_pf = Config(alns_iterations=600, rla_iterations=900,
                        n_runs=1, rla_prefill=pf, early_stop_patience=250)
        gaps, conv = [], []
        for seed in ABL_SEEDS:
            s = RLALNSSolver(inst, cfg_pf)
            plan, hist = s.solve(seed=seed)
            td, _ = plan.gap()
            if td is not None: gaps.append(td)
            thr = hist[-1]*1.01
            ci = next((i for i,c in enumerate(hist) if c<=thr), len(hist)-1)
            conv.append(ci)
        abl_pf.append({'Instance': inst.name, 'Prefill': pf,
                        'Gap%_mean': round(np.mean(gaps),2),
                        'Conv_iter': round(np.mean(conv),0)})

df_abl_pf = pd.DataFrame(abl_pf)
pf_sum = df_abl_pf.groupby('Prefill').agg(
    gap=('Gap%_mean','mean'), conv=('Conv_iter','mean')).reset_index()
print('\n── Ablation 4c: Prefill size ──')
print(pf_sum.round(2).to_string())

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
ax1.plot(pf_sum['Prefill'], pf_sum['gap'], marker='o', lw=2, color='#1d9e75')
ax1.axvline(200, color='orange', ls='--', lw=1.5, label='Default (200)')
ax1.set_xlabel('Prefill episodes'); ax1.set_ylabel('Mean Gap% (↓)')
ax1.set_title('4c: Gap% vs Prefill', fontweight='bold'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(pf_sum['Prefill'], pf_sum['conv'], marker='s', lw=2, color='#5f5fae')
ax2.axvline(200, color='orange', ls='--', lw=1.5, label='Default (200)')
ax2.set_xlabel('Prefill episodes'); ax2.set_ylabel('Iterations to convergence')
ax2.set_title('4c: Convergence vs Prefill', fontweight='bold'); ax2.legend(); ax2.grid(alpha=0.3)
plt.suptitle('Ablation 4c: Prefill Buffer Size', fontweight='bold')
plt.tight_layout(); _savefig('ablation_prefill.png'); print('✅ Saved ablation_prefill.png')

In [ ]:
# ── 23. Final Dashboard & Route Visualisation ─────────────────────────────────
plot_family_summary(df_full)
plot_dashboard(df_full)

# Routes for RC101
for algo, Cls in [('ALNS', ALNSSolver), ('RL-ALNS', RLALNSSolver)]:
    s = Cls(RC1[0], CFG); plan, _ = s.solve(seed=CFG.seed)
    plot_routes(plan)

In [ ]:
# ── 24. NEXUS Export ──────────────────────────────────────────────────────────
import json

MAP_INST = RC1[0]
print(f'Exporting NEXUS demo for {MAP_INST.name}...')

def _solve_export(inst, Cls, label):
    s = Cls(inst, CFG); plan, hist = s.solve(seed=CFG.seed)
    routes = []
    for ri, route in enumerate(plan.routes):
        if not route: continue
        d = float(inst.dist[0, route[0]])
        for i in range(len(route)-1): d += float(inst.dist[route[i], route[i+1]])
        d += float(inst.dist[route[-1], 0])
        routes.append({'id': ri+1, 'nodes': [int(n) for n in route], 'dist': round(d,2)})
    bks_td = BKS[inst.name]['td']
    td = sum(r['dist'] for r in routes)
    gap = round((td-bks_td)/bks_td*100, 2)
    print(f'  {label}: nv={len(routes)}, td={td:.1f}, gap={gap:+.1f}%')
    return {'algo': label, 'nv': len(routes), 'td': round(td,2), 'gap_pct': gap,
            'bks_nv': BKS[inst.name]['nv'], 'bks_td': bks_td,
            'routes': routes, 'history': [round(float(c),2) for c in hist]}

# Solve for export
alns_exp = _solve_export(MAP_INST, ALNSSolver,  'ALNS')
rla_exp  = _solve_export(MAP_INST, RLALNSSolver, 'RL-ALNS')

# Get real op_counts from last RL-ALNS run
_rla_s = RLALNSSolver(MAP_INST, CFG)
_rla_s.solve(seed=CFG.seed)
op_matrix = [[_rla_s.op_counts.get((di,ri),0) for ri in range(N_R)] for di in range(N_D)]

# Nodes
nodes_exp = [{'id':int(i),'x':float(MAP_INST.coords[i,0]),'y':float(MAP_INST.coords[i,1]),
              'demand':float(MAP_INST.demands[i]),'ready':float(MAP_INST.ready_times[i]),
              'due':float(MAP_INST.due_times[i]),'svc':float(MAP_INST.service_times[i])}
             for i in range(MAP_INST.n+1)]

# Summary from df_full using correct column names
summary_exp = []
for _, row in df_full.iterrows():
    summary_exp.append({
        'instance': str(row.get('Instance','')),
        'algo':     str(row.get('Algorithm','')),
        'nv':       float(row.get('NV_mean',0)),
        'td':       float(row.get('TD_mean',0)),
        'gap_pct':  float(row.get('Gap%',0) or 0),
        'cv_nv':    float(row.get('NV_cv',0)),
        'cv_td':    float(row.get('TD_cv',0)),
        'time_s':   float(row.get('Time_s',0)),
    })

# Transfer rows
transfer_exp = []
for _, row in df_transfer.iterrows():
    transfer_exp.append({
        'instance': str(row['Instance']),
        'algo':     str(row['Algorithm']),
        'nv':       float(row['NV_mean']),
        'td':       float(row['TD_mean']),
        'gap_pct':  float(row.get('Gap%',0) or 0),
    })

OUT = {
    'meta': {
        'instance': MAP_INST.name, 'n_customers': int(MAP_INST.n),
        'capacity': float(MAP_INST.capacity), 'horizon': float(MAP_INST.horizon),
        'dataset': 'Solomon Full (56 inst)', 'version': 'v5',
        'algo_desc': {
            'ALNS':     'Adaptive Large Neighbourhood Search (baseline)',
            'RL-ALNS':  'Dueling Double DQN + gating selects operators inside ALNS (proposed)',
            'RL-ALNS★': 'Zero-shot transfer: trained on type-1, applied to type-2',
            'OR-Tools': 'Google OR-Tools GLS (industrial baseline, 60s)',
            'DQN':      'Constructive RL (ablation — motivates hybrid design)',
        }
    },
    'nodes': nodes_exp, 'alns': alns_exp, 'rl_alns': rla_exp,
    'op_matrix': op_matrix,
    'destroy_ops': ['Random','Worst','Shaw','Route','TW-Urgent'],
    'repair_ops':  ['Greedy','Regret-2','Regret-3','TW-Greedy'],
    'summary': summary_exp, 'transfer': transfer_exp,
}

out_path = os.path.join(OUTPUT_DIR, 'nexus_demo.json')
with open(out_path, 'w') as f: json.dump(OUT, f, separators=(',',':'))
print(f'\n✅ nexus_demo.json → {os.path.getsize(out_path)/1024:.1f} KB')
print('   Download from Kaggle Output panel.')